# 🎬 Inferencia sobre Video - Trabajo Final Visión por Computadora
## Pipeline integrado: Detection + Segmentation + Pose

Este notebook carga **3 modelos de YOLO** y los aplica simultáneamente sobre cada frame de un video:

1. **YOLO Segmentation (COCO)** → muestra objetos COCO con máscara de color
2. **YOLO Pose (personas)** → dibuja keypoints y skeleton
3. **YOLO Detection custom (best.pt)** → bboxes con label y confianza para clases nuevas

> Las clases `person` y las clases custom se **excluyen** de la segmentación COCO (las maneja Detection/Pose).

## 1. 📦 Instalación de dependencias

In [ ]:
!pip install -q ultralytics opencv-python numpy torch torchvision Pillow matplotlib

## 2. 📚 Imports y configuración

Importamos las librerías y agregamos `inference/` al path para usar `utils.py`.

In [ ]:
import sys
import os
import time
from pathlib import Path

# Agregar inference/ al path para importar utils
sys.path.insert(0, 'inference')

import cv2
import numpy as np
from ultralytics import YOLO

import utils

print("✅ Librerías importadas correctamente")

## 3. 🧠 Cargar los 3 modelos

Se cargan los 3 modelos en paralelo (los archivos `.pt` se descargan automáticamente la primera vez).

> Si tu versión de ultralytics no tiene `yolo26n-seg.pt` / `yolo26n-pose.pt`, usar las versiones disponibles (`yolo11n-seg.pt`, etc.).

In [ ]:
# Ajustar los nombres de los modelos segun la version disponible
SEG_MODEL_NAME    = 'yolo26n-seg.pt'    # COCO segmentation
POSE_MODEL_NAME   = 'yolo26n-pose.pt'   # pose estimation
CUSTOM_MODEL_PATH = 'best.pt'           # detector custom (entregable del training)

print("Cargando modelos...")
seg_model    = YOLO(SEG_MODEL_NAME)
pose_model   = YOLO(POSE_MODEL_NAME)
custom_model = YOLO(CUSTOM_MODEL_PATH)
print("✅ 3 modelos cargados:")
print(f"  - Segmentacion:  {SEG_MODEL_NAME}")
print(f"  - Pose:          {POSE_MODEL_NAME}")
print(f"  - Custom:        {CUSTOM_MODEL_PATH} ({list(custom_model.names.values())})")

## 4. 🎥 Configurar el video de entrada y salida

Apuntá `VIDEO_PATH` a uno de los videos cortos (~20s) en `videos/input/`.

In [ ]:
from pathlib import Path

VIDEO_PATH = 'videos/input/video1.mp4'  # <-- Cambiar al video que quieras
OUTPUT_PATH = 'videos/output/resultado_video1.mp4'

# Crear carpeta de salida si no existe
Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise FileNotFoundError(f"❌ No se pudo abrir {VIDEO_PATH}\nVerifica que exista y que el codec sea compatible.")

w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Video: {VIDEO_PATH}")
print(f"  Resolucion: {w}x{h}")
print(f"  FPS: {fps:.2f}")
print(f"  Frames totales: {total_frames}")
print(f"  Duracion: {total_frames/fps:.1f}s")

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (w, h))
print(f"\nVideo de salida: {OUTPUT_PATH}")

## 5. 🎞️ Loop de procesamiento frame a frame

Aplica los 3 modelos a cada frame y combina los resultados en un único frame anotado.

In [ ]:
CONFIDENCE = 0.4
CUSTOM_CLASSES = list(custom_model.names.values())  # ej: ['mate', 'termo', 'factura']
MAX_FRAMES = None  # poner un int para probar con N frames, o None para procesar todo

frame_count = 0
t_start = time.time()

try:
    while cap.isOpened():
        ok, frame = cap.read()
        if not ok:
            break

        frame_count += 1
        if MAX_FRAMES is not None and frame_count > MAX_FRAMES:
            break

        # 1) Segmentacion COCO (excluye 'person' y clases custom)
        seg_res = seg_model(frame, conf=CONFIDENCE, verbose=False)[0]

        # 2) Pose estimation (personas)
        pose_res = pose_model(frame, conf=CONFIDENCE, verbose=False)[0]

        # 3) Deteccion de clases custom
        cust_res = custom_model(frame, conf=CONFIDENCE, verbose=False)[0]

        # 4) Combinar todo en un unico frame
        annotated, counts = utils.annotate_frame(
            frame=frame,
            seg_result=seg_res,
            pose_result=pose_res,
            custom_result=cust_res,
            seg_model_names=seg_model.names,
            pose_model_names=pose_model.names,
            custom_model_names=custom_model.names,
            custom_class_names=CUSTOM_CLASSES,
            conf_threshold=CONFIDENCE,
        )

        # 5) Calcular FPS y overlay
        elapsed = time.time() - t_start
        current_fps = frame_count / elapsed if elapsed > 0 else 0
        annotated = utils.add_info_overlay(
            annotated, current_fps,
            custom_count=counts['n_custom'],
            seg_count=counts['n_seg'],
            person_count=counts['n_persons'],
        )

        out.write(annotated)

        if frame_count % 30 == 0:
            print(f"  Frame {frame_count}/{total_frames} - FPS: {current_fps:.1f}")

finally:
    cap.release()
    out.release()

elapsed = time.time() - t_start
print(f"\n✅ Video procesado: {frame_count} frames en {elapsed:.1f}s ({frame_count/elapsed:.1f} FPS promedio)")
print(f"   Guardado en: {OUTPUT_PATH}")

## 6. 📊 Verificar el resultado

Muestreo de un par de frames del video procesado para verificar visualmente la salida.

In [ ]:
# Leer 3 frames espaciados del video de salida para mostrarlos
out_cap = cv2.VideoCapture(OUTPUT_PATH)
n = int(out_cap.get(cv2.CAP_PROP_FRAME_COUNT))
samples = [0, n // 2, n - 1] if n >= 3 else list(range(n))

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(samples), figsize=(18, 5))
if len(samples) == 1:
    axes = [axes]

for ax, idx in zip(axes, samples):
    out_cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, f = out_cap.read()
    if ok:
        ax.imshow(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
        ax.set_title(f"Frame {idx}")
    ax.axis('off')

plt.tight_layout()
plt.show()
out_cap.release()
print("Para reproducir el video completo:", OUTPUT_PATH)